# 📊 FinDataMining
Notebook 01: **Extracción de Datos**

---

## Preliminares

* Importar librerías:

In [ ]:
import pandas as pd
from src.config import data_folder

%load_ext autoreload
%autoreload 2
from src.extract import *

## Extracción de datos financieros de `simFin`

Luego de haber creado la cuenta gratuita e ingresado la clave API en el repositorio local (ver [README.md](README.md)), se obtienen de `simFin` los datos financieros históricos de un período máximo de 5 años, con un lag de 1 año. A partir de los datos descargados, se genera la lista de tickers en el universo del proyecto, según los criterios:

- Umbral de filas: cantidad mínima de reportes trimestrales disponibles (por defecto 19).

- Columna ranking: se calcula el promedio de la columna indicada (por defecto TotalRevenue).

- Cantidad de tickers: luego de rankear según la columna indicada, se toman los top n tickers (por defecto 550).

In [2]:
df_simfin_unfiltered = extraer_simfin()
tickers_universe_list = generar_universo_tickers(df_simfin_unfiltered)

# Se agrega columna 'FinancialsSource' para indicar que los datos provienen de simFin
df_simfin_unfiltered['FinancialsSource'] = 'simFin'

print("Datos extraidos de simFin, dimensiones:", df_simfin_unfiltered.shape)

Dataset "us-income-quarterly" on disk (6 days old).
- Loading from disk ... Done!
Dataset "us-balance-quarterly" on disk (7 days old).
- Loading from disk ... Done!
Dataset "us-cashflow-quarterly" on disk (6 days old).
- Loading from disk ... Done!
Dataset "us-income-banks-quarterly" on disk (6 days old).
- Loading from disk ... Done!
Dataset "us-balance-banks-quarterly" on disk (6 days old).
- Loading from disk ... Done!
Dataset "us-cashflow-banks-quarterly" on disk (6 days old).
- Loading from disk ... Done!
Dataset "us-income-insurance-quarterly" on disk (6 days old).
- Loading from disk ... Done!
Dataset "us-balance-insurance-quarterly" on disk (6 days old).
- Loading from disk ... Done!
Dataset "us-cashflow-insurance-quarterly" on disk (6 days old).
- Loading from disk ... Done!
Datos extraidos de simFin, dimensiones: (50230, 83)


## Extracción de precios de `yfinance`

* Extraer precios de los tickers y del índice SPY (demora unos minutos):

In [3]:
df_prices = extraer_precios(tickers_universe_list)

# Se extrae precio del Índice de Mercado para usar en cálculos y se guarda en fichero 
df_index = extraer_precios(['SPY'])
df_index.to_parquet(f"{data_folder}/market_index.parquet")

print("Extraídos precios del universo de tickers y del indice.")
print("Dimensiones de df_precios:", df_prices.shape)
print("Dimensiones de df_index:", df_index.shape)

Iniciando descarga masiva para 549 tickers únicos...


[*********************100%***********************]  549 of 549 completed

27 Failed downloads:
['APY', 'SKX', 'HES', 'GMS', 'FLT', 'SPR', 'IPG', 'MDC', 'K', 'RYI', 'HOUS', 'X', 'WBA', 'JNPR', 'PDCO', 'TM-28', 'QRTEA', 'FI', 'FL', 'JWN', 'CCH', 'SQ', 'DISH', 'HBI', 'TPX', 'SAVE']: YFPricesMissingError('possibly delisted; no price data found  (period=6y) (Yahoo error = "No data found, symbol may be delisted")')
['QVC']: YFPricesMissingError('possibly delisted; no price data found  (period=6y)')


Reestructurando los datos...
Extracción completada.
Iniciando descarga masiva para 1 tickers únicos...


[*********************100%***********************]  1 of 1 completed

Reestructurando los datos...
Extracción completada.
Extraídos precios del universo de tickers y del indice.
Dimensiones de df_precios: (13028, 5)
Dimensiones de df_index: (25, 5)


* Actualizar el universo de tickers con los que se obtuvieron precios:

In [4]:
tickers_list_updated = df_prices['Ticker'].unique().tolist()

print("Actualizado el universo a tickers con precios disponibles en yfinance.")
print("Cantidad de tickers actualizado:", len(tickers_list_updated))

Actualizado el universo a tickers con precios disponibles en yfinance.
Cantidad de tickers actualizado: 522


## Extracción de datos complementarios

Se obtiene la lista de tickers del S&P 500 desde el fichero `constituents.csv`:

* De dicho fichero se obtiene para los tickers que pertenecen al índice, la fecha en la que fueron agregados con la columna `DataAdded`.
* Si no existe el fichero, lo descarga desde GitHub.
* Para actualizar la lista de componentes, cambiar `force_update` a True, ejecutar y volver a dejar en False.

In [5]:
# Obtener constituents del indice S&P 500
ruta_sp500 = descargar_constituents_sp(force_update=False) 

# Cargar y limpiar datos de constituents
print("Cargando los tickers del S&P 500.")
df_tickers_sp_raw = pd.read_csv(ruta_sp500)
df_tickers_sp = limpiar_constituents_sp(df_tickers_sp_raw)
print("Cantidad de tickers:", df_tickers_sp['Ticker'].nunique())
df_tickers_sp.info()

Usando archivo constituents.csv local.
Cargando los tickers del S&P 500.
Cantidad de tickers: 503
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 503 entries, 0 to 502
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Ticker     503 non-null    object
 1   DateAdded  503 non-null    object
dtypes: object(2)
memory usage: 8.0+ KB


* Unir dataframes de precios y de tickers del S&P 500 para obtener la columna 'DateAdded' (fecha en que se comenzo a incluir en el índice):

In [6]:
df_merged = pd.merge(
    df_prices,
    df_tickers_sp,
    on= 'Ticker',
    how= 'left'
)
print("Unidos precios con DateAdded del indice S&P500, dimensiones del dataset:", df_merged.shape)
df_merged.info()

Unidos precios con DateAdded del indice S&P500, dimensiones del dataset: (13028, 6)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13028 entries, 0 to 13027
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Date       13028 non-null  datetime64[ns]
 1   Ticker     13028 non-null  object        
 2   Close      13028 non-null  float64       
 3   Open       13028 non-null  float64       
 4   Volume     13028 non-null  float64       
 5   DateAdded  7567 non-null   object        
dtypes: datetime64[ns](1), float64(3), object(2)
memory usage: 610.8+ KB


* Extraer de `yfinance` la información de sector e industria (demora unos minutos):

In [7]:
df_info = extraer_info(tickers_list_updated)
df_info.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 522 entries, 0 to 521
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Ticker    522 non-null    object
 1   Sector    522 non-null    object
 2   Industry  522 non-null    object
dtypes: object(3)
memory usage: 12.4+ KB


* Se une el dataframe de info con con df_merged (precios + s&p500):

In [8]:
df_with_info = pd.merge(
    df_merged,
    df_info,
    on= 'Ticker',
    how= 'left'
)
df_with_info.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13028 entries, 0 to 13027
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Date       13028 non-null  datetime64[ns]
 1   Ticker     13028 non-null  object        
 2   Close      13028 non-null  float64       
 3   Open       13028 non-null  float64       
 4   Volume     13028 non-null  float64       
 5   DateAdded  7567 non-null   object        
 6   Sector     13028 non-null  object        
 7   Industry   13028 non-null  object        
dtypes: datetime64[ns](1), float64(3), object(4)
memory usage: 814.4+ KB


## Extracción de  datos financieros de `yfinance`

Se descargan los últimos 4 reportes trimestrales.

`yfinance` no incluye la fecha de publicación, sólo la fecha de los Estados Financieros. Para evitar el "LookAhead" bias, se ofrecen 2 alternativas con el parametro `aproximar_fechas`:

*aproximar_fechas = True*: 
* Se estima la fecha de publicación con una regla de 30 días por defecto (editable en src/config.py). 
* *Demora de ejecución* de aproximadamente 10 minutos.

*aproximar_fechas = False*: 
* Obtiene la fecha real de publicación usando yf.earnings_dates. 
* Se aplica la regla de 30 días sólo si no se encuentra. 
* Puede demorar 30 minutos o más.

In [9]:
print("Extrayendo datos financieros de yfinance, demora varios minutos.")
df_yfinance, tickers_sin_datos = extraer_financials(tickers_list_updated, aproximar_fechas = True)

# Se agrega columna 'FinancialsSource' para indicar que los datos provienen de yfinance
df_yfinance['FinancialsSource'] = 'yfinance'
df_yfinance.info()

Extrayendo datos financieros de yfinance, demora varios minutos.
Sin datos financieros trimestrales para CNLPL
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2084 entries, 0 to 2083
Data columns (total 23 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   Date                           2084 non-null   datetime64[ns]
 1   Total Revenue                  2079 non-null   float64       
 2   Gross Profit                   1950 non-null   float64       
 3   Operating Income               1964 non-null   float64       
 4   Net Income                     2079 non-null   float64       
 5   EBITDA                         1964 non-null   float64       
 6   Basic Average Shares           2039 non-null   float64       
 7   Cash And Cash Equivalents      2074 non-null   float64       
 8   Current Debt                   1563 non-null   float64       
 9   Long Term Debt                 1955 non-

## Unión de datasets y almacenamiento

* Se unen los datos financieros de `simFin` y `yfinance`:

In [10]:
# Se convierte la lista de tickers sin datos de yfinance a un set
sin_datos_set = set(tickers_sin_datos)

# Actualizar la lista de tickers en el universo y guardarlo en el fichero tickers_universe.csv
tickers_list_updated = [ticker for ticker in tickers_list_updated if ticker not in sin_datos_set]
guardar_fichero_tickers(tickers_list_updated)
print(f"Universo de tickers guardado en fichero. Total: {len(tickers_list_updated)} tickers.")

# Filtrar los tickers actualizados en los datos de simFin
df_simfin = df_simfin_unfiltered[df_simfin_unfiltered['Ticker'].isin(tickers_list_updated)]

# Definir columnas a mantener en simFin para que coincidan y estandarizar antes de unir
cols_yfinance = df_yfinance.columns.to_list()
df_simfin_clean = estandarizar_simfin(df_simfin, cols_yfinance)

df_financials_completo = unir_financials(df_yfinance, df_simfin_clean)
df_financials_completo.info()

Universo de tickers guardado en fichero. Total: 521 tickers.
Se han encontrado 49 filas con Ticker y Date solapados (presentes en ambas fuentes).
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11625 entries, 0 to 11624
Data columns (total 23 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   Ticker                         11625 non-null  object        
 1   Date                           11609 non-null  datetime64[ns]
 2   Total Revenue                  11605 non-null  float64       
 3   Gross Profit                   10633 non-null  float64       
 4   Operating Income               11490 non-null  float64       
 5   Net Income                     11605 non-null  float64       
 6   EBITDA                         1910 non-null   float64       
 7   Basic Average Shares           11558 non-null  float64       
 8   Cash And Cash Equivalents      11570 non-null  float64       
 9   Cur

* Unir dataframe de precios con datos financieros:

In [11]:
df_final = pd.merge(
    df_with_info, 
    df_financials_completo, 
    on=['Date', 'Ticker'],
    how='left'
)
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13028 entries, 0 to 13027
Data columns (total 29 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   Date                           13028 non-null  datetime64[ns]
 1   Ticker                         13028 non-null  object        
 2   Close                          13028 non-null  float64       
 3   Open                           13028 non-null  float64       
 4   Volume                         13028 non-null  float64       
 5   DateAdded                      7567 non-null   object        
 6   Sector                         13028 non-null  object        
 7   Industry                       13028 non-null  object        
 8   Total Revenue                  11482 non-null  float64       
 9   Gross Profit                   10514 non-null  float64       
 10  Operating Income               11367 non-null  float64       
 11  Net Income     

* Limpieza final y almacenamiento de datos en fichero raw_data.parquet:

In [12]:
df_final_clean = limpieza_final(df_final)
df_final_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11367 entries, 0 to 11366
Data columns (total 29 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   Date                         11367 non-null  datetime64[ns]
 1   Ticker                       11367 non-null  object        
 2   Close                        11367 non-null  float64       
 3   Open                         11367 non-null  float64       
 4   Volume                       11367 non-null  float64       
 5   DateAdded                    6700 non-null   object        
 6   Sector                       11367 non-null  object        
 7   Industry                     11367 non-null  object        
 8   TotalRevenue                 11367 non-null  float64       
 9   GrossProfit                  10514 non-null  float64       
 10  OperatingIncome              11367 non-null  float64       
 11  NetIncome                    11367 non-nu

In [13]:
# Guardar datos extraidos en fichero raw_data
df_final_clean.to_parquet(f"{data_folder}/raw_data.parquet")
print(f"Extraccion finalizada.\nFichero 'raw_data.parquet' guardado en la carpeta {data_folder}")

Extraccion finalizada.
Fichero 'raw_data.parquet' guardado en la carpeta data
